In [1]:
import os
os.environ["POLYGON_TOKEN"] = "Qsv1DasKgXC2pTUqtKRqSnMH5RwgpBob"
os.environ["POLY_PATH_LOCAL"] = "/mnt/c/Users/Public/aae/local"
os.environ["POLY_PATH_INPUT"] = "/mnt/c/Users/Public/aae/input"
os.environ["POLY_PATH_TEMP"] = "!/temp"
os.environ["POLY_NAS_CACHE"] = "/mnt/c/Users/Public/aae/nas"


from cdxcore.subdir import SubDir 
import polars as pl
from poly.poly import PolyHandle
from datetime import date

temp = SubDir("!/temp")

@temp.cache("0.3", uid=lambda ticker, d : f"{ticker}/{d:%Y-%m-%d}")
def fetch_open_close(ticker: str, d: date, adjusted: bool, *, poly: PolyHandle | None = None) -> dict:
    """
    /v1/open-close/{ticker}/{YYYY-MM-DD}?adjusted=adjusted
    Asserts presence of open/close/volume.
    """
    poly = PolyHandle() if poly is None else poly
    url = f"{PolyHandle.BASE_URL}/v1/open-close/{ticker}/{d:%Y-%m-%d}"
    data = poly(url, {"adjusted": "true" if adjusted else "false"})
    # hard asserts as requested
    assert isinstance(data, dict), f"open-close returned non-dict for {ticker} on {d}: {type(data)}"
    assert "open" in data and "close" in data and "volume" in data, f"open-close missing keys for {ticker} on {d}: keys={list(data.keys())}"
    assert data["open"] is not None and data["close"] is not None and data["volume"] is not None, f"open-close has nulls for {ticker} on {d}: open={data.get('open')} close={data.get('close')} volume={data.get('volume')}"
    return pl.DataFrame(data)

_ = fetch_open_close("SPY", date(2025, 3, 18), adjusted=True, override_cache_mode="off")
print(fetch_open_close.cache_info.last_cached  )
print(fetch_open_close.cache_info.unique_id)
print(_)

fetch_open_close.version


Added '/mnt/c/users/hans/onedrive/python3/packages/cdxcore' to the import path
Added '/mnt/c/users/hans/onedrive/python3/packages/cdxjax' to the import path
False
SPY/2025-03-18
shape: (1, 10)
┌────────┬────────────┬────────┬───────┬───┬────────┬─────────────┬────────────┬───────────┐
│ status ┆ from       ┆ symbol ┆ open  ┆ … ┆ close  ┆ volume      ┆ afterHours ┆ preMarket │
│ ---    ┆ ---        ┆ ---    ┆ ---   ┆   ┆ ---    ┆ ---         ┆ ---        ┆ ---       │
│ str    ┆ str        ┆ str    ┆ f64   ┆   ┆ f64    ┆ f64         ┆ f64        ┆ f64       │
╞════════╪════════════╪════════╪═══════╪═══╪════════╪═════════════╪════════════╪═══════════╡
│ OK     ┆ 2025-03-18 ┆ SPY    ┆ 564.8 ┆ … ┆ 561.02 ┆ 6.6041422e7 ┆ 561.1999   ┆ 565.93    │
└────────┴────────────┴────────┴───────┴───┴────────┴─────────────┴────────────┴───────────┘


0.3